# Predictive Maintenance — AI4I 2020 Dataset: Preprocessing & Exploratory Data Analysis (EDA)

**Project:** Machine Learning System for Early Detection of Equipment Failure  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset  
**Target:** `Machine failure` (Binary: 0 = Normal Operation, 1 = Machine Failure)  

---

### Objectives
1. **Workspace & Raw Data Audit:** Inspect the dataset schema, data types, missing values, duplicates, and physical units.
2. **Data Quality & Outlier Diagnostics:** Audit statistical outliers via Interquartile Range (IQR) and examine their relationship to actual failure events.
3. **Exploratory Visualizations:** Distribution plots, correlation heatmap, sensor comparisons stratified by failure status, and pair plots.
4. **Target & Class Imbalance:** Analyze the severe ~28.5:1 imbalance and failure modes breakdown (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`), noting data leakage risks.
5. **Physics-Based Feature Engineering Analysis:** Evaluate temperature differential ($\Delta T$), mechanical power ($P_{	ext{mech}}$), and overstrain index ($I_{	ext{overstrain}}$).
6. **Stratified Train-Test Splitting:** Create an 80:20 stratified split on the 5 primary sensor features preserving UDI index traceability for model training.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# High-resolution plotting configuration
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.edgecolor': '#CBD5E1',
    'axes.linewidth': 1.1,
    'grid.color': '#E2E8F0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7
})

print("Environment libraries imported successfully.")


## 1. Load and Verify the Raw Dataset


In [ ]:
# Locate the raw dataset
candidate_paths = [
    Path("../data/raw/ai4i2020.csv"),
    Path("../../ai4i2020.csv"),
    Path("ml/data/raw/ai4i2020.csv"),
    Path("ai4i2020.csv")
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Raw ai4i2020.csv not found!")

print(f"Loading raw dataset from: {data_path.resolve()}")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


## 2. Comprehensive Data Quality Checks
- Checking missing values across all columns
- Checking duplicate records (exact rows, UDI keys, Product IDs)
- Verifying data types and non-finite values


In [ ]:
print("--- Missing Values ---")
missing_info = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
display(missing_info)

print("\n--- Duplicate Analysis ---")
print(f"Exact Duplicate Rows: {df.duplicated().sum()}")
print(f"Duplicate UDI entries: {df['UDI'].duplicated().sum()}")
print(f"Unique Product IDs:   {df['Product ID'].nunique()}")

print("\n--- Data Types ---")
display(df.dtypes)


## 3. Parametric & Non-Parametric Descriptive Statistics


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
stats_df = df[num_cols].describe().T
stats_df['skewness'] = df[num_cols].skew()
stats_df['kurtosis'] = df[num_cols].kurtosis()
stats_df


## 4. Target Variable Analysis: Severe Class Imbalance
`Machine failure` is our primary classification target:
- `0`: Normal operational condition
- `1`: Machine failure (breakdown)


In [ ]:
target_counts = df['Machine failure'].value_counts().sort_index()
total_records = len(df)
n_normal = target_counts[0]
n_failure = target_counts[1]
ratio = n_normal / n_failure

print(f"Normal Operations (0): {n_normal:,} ({n_normal / total_records * 100:.2f}%)")
print(f"Machine Failures  (1): {n_failure:,} ({n_failure / total_records * 100:.2f}%)")
print(f"Imbalance Ratio:       {ratio:.2f} : 1")

plt.figure(figsize=(7, 4.5))
bars = plt.bar(['Normal (0)', 'Failure (1)'], [n_normal, n_failure], color=['#10B981', '#EF4444'], width=0.45, edgecolor='#1E293B')
plt.title("Machine Failure Class Imbalance Distribution", fontweight='bold', pad=12)
plt.ylabel("Number of Records", fontweight='bold')
plt.grid(axis='y')

for b in bars:
    h = b.get_height()
    plt.annotate(f"{h:,}\n({h/total_records*100:.2f}%)", xy=(b.get_x() + b.get_width()/2, h),
                 xytext=(0, 5), textcoords="offset points", ha="center", va="bottom", fontweight="bold")
plt.show()


## 5. Failure Modes Breakdown & Data Leakage Prevention
The dataset contains five specific failure modes:
1. `TWF`: Tool Wear Failure
2. `HDF`: Heat Dissipation Failure
3. `PWF`: Power Failure
4. `OSF`: Overstrain Failure
5. `RNF`: Random Failure

> **CRITICAL WARNING:** These failure mode indicators are **NOT** input features. They are diagnostic failure sub-labels. Including them as model inputs causes catastrophic **data leakage**, as knowing that a failure mode occurred directly gives away the `Machine failure` label.


In [ ]:
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
fm_counts = df[failure_cols].sum()

print("Failure Mode Counts:")
print(fm_counts)

# Check overlap between failure modes and Machine failure
fm_any = (df[failure_cols].sum(axis=1) > 0).astype(int)
print(f"\nRecords with any failure mode flagged: {fm_any.sum()}")
print(f"Machine failure == 1 with NO mode flagged: {((df['Machine failure'] == 1) & (fm_any == 0)).sum()} (unexplained failures)")
print(f"Mode flagged with Machine failure == 0: {((df['Machine failure'] == 0) & (fm_any == 1)).sum()} (RNF cases not tripping shutdown)")


## 6. Primary 5 Sensor Features & Outlier Analysis
The primary model uses strictly the 5 continuous physical sensor readings matching the web interface:
1. `Air temperature [K]`
2. `Process temperature [K]`
3. `Rotational speed [rpm]`
4. `Torque [Nm]`
5. `Tool wear [min]`


In [ ]:
PRIMARY_FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

# IQR Outlier Analysis
outlier_summary = []
for col in PRIMARY_FEATURES:
    q25 = df[col].quantile(0.25)
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25
    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    count = mask.sum()
    failures_in_outliers = df.loc[mask, 'Machine failure'].sum()
    outlier_summary.append({
        'Feature': col,
        'Lower Bound': lower,
        'Upper Bound': upper,
        'Outlier Count': count,
        'Outlier %': (count / len(df)) * 100,
        'Failures in Outliers': failures_in_outliers,
        'Failure Rate in Outliers (%)': (failures_in_outliers / count * 100) if count > 0 else 0
    })

display(pd.DataFrame(outlier_summary))


### Outlier Treatment Decision
Notice that:
- For `Torque [Nm]`, **63.8%** of the IQR statistical outliers are actual machine failures.
- For `Rotational speed [rpm]`, **26.3%** of the IQR statistical outliers are actual machine failures.

**Conclusion:** Statistical outliers must **NOT** be pruned or clipped in predictive maintenance. The extreme physical readings (e.g., motor stalls, extreme torque resistance, runaway spindle speeds) are the exact physical anomalies preceding equipment breakdown.


## 7. Sensor Distributions Stratified by Operational State


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(PRIMARY_FEATURES):
    ax = axes[i]
    sns.kdeplot(data=df[df['Machine failure'] == 0], x=col, ax=ax, label='Normal (0)',
                color='#10B981', fill=True, alpha=0.35, linewidth=2)
    sns.kdeplot(data=df[df['Machine failure'] == 1], x=col, ax=ax, label='Failure (1)',
                color='#EF4444', fill=True, alpha=0.45, linewidth=2)
    ax.set_title(col, fontweight='bold')
    ax.legend()
    ax.grid(True)

# 6th plot: Torque vs Speed scatter showing failure boundary
ax_sc = axes[5]
sns.scatterplot(data=df[df['Machine failure'] == 0], x='Rotational speed [rpm]', y='Torque [Nm]',
                ax=ax_sc, color='#10B981', alpha=0.2, s=15, label='Normal (0)')
sns.scatterplot(data=df[df['Machine failure'] == 1], x='Rotational speed [rpm]', y='Torque [Nm]',
                ax=ax_sc, color='#EF4444', alpha=0.85, s=30, edgecolor='black', label='Failure (1)')
ax_sc.set_title('Torque vs Rotational Speed (Failure Boundaries)', fontweight='bold')
ax_sc.legend()
ax_sc.grid(True)

plt.tight_layout()
plt.show()


## 8. Correlation Analysis


In [ ]:
cols_to_corr = PRIMARY_FEATURES + ['Machine failure']
corr = df[cols_to_corr].corr(method='pearson')

mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='vlag', vmin=-1, vmax=1,
            linewidths=1, linecolor='white')
plt.title('Pearson Correlation Heatmap: Sensor Features & Machine Failure', fontweight='bold', pad=12)
plt.show()


## 9. Physics-Based Feature Engineering Exploration
Evaluating 3 physically motivated derived features:
1. **Temperature Difference ($\Delta T$):** $	ext{Process temp} - 	ext{Air temp}$ ($< 8.6$ K at low speed triggers HDF).
2. **Mechanical Power ($P_{	ext{mech}}$):** $	au \cdot \omega \cdot rac{2\pi}{60}$ ($< 3500$ W or $> 9000$ W triggers PWF).
3. **Overstrain Index ($I_{	ext{overstrain}}$):** $	au \cdot 	ext{Tool wear}$ ($> 11000$ Nm$\cdot$min triggers OSF).

> **Note:** These are evaluated experimentally here. For the primary model baseline, we strictly prepare the 5 raw sensor inputs.


In [ ]:
df_eng = df.copy()
df_eng['delta_temperature'] = df_eng['Process temperature [K]'] - df_eng['Air temperature [K]']
df_eng['mechanical_power'] = df_eng['Torque [Nm]'] * df_eng['Rotational speed [rpm]'] * 2 * np.pi / 60
df_eng['overstrain_index'] = df_eng['Torque [Nm]'] * df_eng['Tool wear [min]']

eng_corrs = df_eng[['delta_temperature', 'mechanical_power', 'overstrain_index', 'Machine failure']].corr()['Machine failure']
print("Correlation with Machine failure:")
print(eng_corrs)


## 10. Stratified Train-Test Splitting (80:20)
We perform an 80:20 stratified train-test split:
- **Features ($X$):** 5 primary sensor inputs (indexed by `UDI` for complete auditability).
- **Target ($y$):** `Machine failure`.
- **Stratification:** Preserves the ~3.39% failure rate identically in both partitions.
- **Random State:** 42.


In [ ]:
X = df.set_index('UDI')[PRIMARY_FEATURES]
y = df.set_index('UDI')['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

print("\nClass distribution in Training Set:")
print(f"Normal (0): {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.2f}%)")
print(f"Failure (1): {(y_train == 1).sum():,} ({(y_train == 1).sum()/len(y_train)*100:.2f}%)")

print("\nClass distribution in Testing Set:")
print(f"Normal (0): {(y_test == 0).sum():,} ({(y_test == 0).sum()/len(y_test)*100:.2f}%)")
print(f"Failure (1): {(y_test == 1).sum():,} ({(y_test == 1).sum()/len(y_test)*100:.2f}%)")


## 11. Pipeline Verification & Ready for Model Training
- Zero missing values.
- Legitimate failure outliers preserved.
- No data leakage from failure mode sub-labels.
- 5 primary inputs match web frontend form.
- Reproducible split with identical class distribution.

The data is fully preprocessed and ready for downstream model evaluation (Random Forest, Gradient Boosting, etc.) with class-imbalance aware metrics (PR-AUC, F1, Recall).
